# Teaching a 7B Model to Beat GPT-4o at Chess

In this notebook, we'll train **Qwen 2.5 7B Instruct** to play chess using:
1. **Supervised Fine-Tuning (SFT)** — teach the model chess move format and basic strategy
2. **GRPO (Group Relative Policy Optimization)** — reinforcement learning with Stockfish as the reward signal

All heavy compute runs on [Modal](https://modal.com) serverless GPUs. This notebook runs locally and dispatches jobs.

**Prerequisites:**
- Modal CLI installed and authenticated (`pip install modal && modal setup`)
- `.env` file with `OPENAI_API_KEY`, `HF_TOKEN`, `WANDB_API_KEY`

**Estimated cost:** ~$8-10 for the full pipeline (A100-80GB at $2.50/hr)

## 1. Setup & Prerequisites

In [ ]:
# Install local dependencies
!uv pip install python-chess openai modal

In [ ]:
# Check Modal is authenticated
!modal setup 2>/dev/null; modal profile current

In [ ]:
# Create a single Modal secret with all API keys from .env
# (--force overwrites if it already exists)
!source .env && modal secret create rl-chess-secrets \
    "WANDB_API_KEY=$WANDB_API_KEY" \
    "HF_TOKEN=$HF_TOKEN" \
    "OPENAI_API_KEY=$OPENAI_API_KEY" \
    --force

In [ ]:
# Verify secrets exist
!modal secret list

## 2. Understanding the Chess Environment

Before training, let's understand how we represent chess positions and moves.
We use:
- **FEN** (Forsyth-Edwards Notation) — compact string representation of a board position
- **UCI** (Universal Chess Interface) — move format like `e2e4` (from-square to-square)

In [ ]:
import chess

# Create the starting position
board = chess.Board()
print("Starting position FEN:")
print(board.fen())
print()
print(board)

In [ ]:
# Legal moves in UCI format — this is what we'll feed to the model
legal_moves = [m.uci() for m in board.legal_moves]
print(f"{len(legal_moves)} legal moves: {' '.join(legal_moves)}")

In [ ]:
# Make some moves and see how the board changes
board.push(chess.Move.from_uci("e2e4"))  # King's pawn
board.push(chess.Move.from_uci("e7e5"))  # Symmetric response
board.push(chess.Move.from_uci("g1f3"))  # Knight out
print(board)
print(f"\nFEN: {board.fen()}")
print(f"Legal moves for Black: {' '.join(m.uci() for m in board.legal_moves)}")

In [ ]:
# Validate moves — this is how the reward function checks legality
test_board = chess.Board(board.fen())

good_move = chess.Move.from_uci("b8c6")  # Knight to c6 (legal)
bad_move = chess.Move.from_uci("a1a2")   # Rook move (illegal for Black)

print(f"b8c6 is legal: {good_move in test_board.legal_moves}")
print(f"a1a2 is legal: {bad_move in test_board.legal_moves}")

## 3. Building the Reward Function

The reward function is the **heart of GRPO training**. It scores each model output on a progressive scale:

| Component | Weight | What it checks |
|-----------|--------|----------------|
| `format_reward` | 0.1 | Did the model use `<move>` tags? |
| `legality_reward` | 0.4 | Is the move legal in this position? |
| `quality_reward` | 0.5 | How good is the move (Stockfish eval)? |

This graduated approach is critical — the model starts knowing nothing about chess.
With binary rewards, it would get zero signal. With progressive rewards,
even outputting the right format gets *some* reward, enabling learning.

In [ ]:
import re

def extract_move(text):
    """Extract UCI move from <move>...</move> tags."""
    match = re.search(r"<move>\s*([a-h][1-8][a-h][1-8][qrbn]?)\s*</move>", text)
    return match.group(1) if match else None

# Test with various outputs
test_outputs = [
    "I think the best move is <move>e2e4</move>",        # Good format
    "The move is e2e4",                                   # Missing tags
    "<move>xyz123</move>",                                 # Bad UCI format
    "<move>a1a2</move>",                                   # Valid UCI, may be illegal
]

for output in test_outputs:
    move = extract_move(output)
    print(f"  {output[:50]:50s} -> {move}")

In [ ]:
# Simulate the full reward pipeline on a position
import math

test_fen = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1"
test_board = chess.Board(test_fen)
print(f"Position: {test_fen}")
print(f"Legal moves: {' '.join(m.uci() for m in test_board.legal_moves)}")
print()

test_completions = [
    "<move>e7e5</move>",          # Legal, common response
    "<move>a7a1</move>",          # Valid UCI, but illegal
    "I play e7e5",                # Wrong format
    "<move>d7d5</move>",          # Legal, Scandinavian Defense
]

for comp in test_completions:
    move_str = extract_move(comp)
    fmt = 1.0 if move_str else 0.0
    legal = 0.0
    quality = 0.0
    if move_str:
        try:
            move = chess.Move.from_uci(move_str)
            legal = 1.0 if move in test_board.legal_moves else 0.0
        except ValueError:
            pass
    total = 0.1 * fmt + 0.4 * legal + 0.5 * quality
    print(f"  {comp:30s}  fmt={fmt:.0f}  legal={legal:.0f}  quality={quality:.1f}  total={total:.2f}")

## 4. Preparing Training Data

We use the [Lichess chess puzzles dataset](https://huggingface.co/datasets/Lichess/chess-puzzles) — ~4.5M puzzles with verified best moves from Stockfish analysis.

The data prep script:
1. Loads puzzles rated 1200-2200 (intermediate difficulty)
2. Extracts the puzzle position (after opponent's setup move)
3. Creates **15K examples for SFT** (FEN + best move pairs)
4. Creates **5K examples for GRPO** (FEN positions only — the model discovers good moves via RL)

This runs on a CPU-only container (no GPU needed).

In [ ]:
# Run data preparation on Modal (CPU only, ~2-3 min)
!modal run scripts/modal_app.py::run_data_prep

## 5. SFT Warmstart

SFT (Supervised Fine-Tuning) teaches the model **how** to play chess — specifically:
- The prompt format (FEN + legal moves)
- The output format (`<move>...</move>` tags)
- Basic move selection from high-quality games

We use **LoRA** (Low-Rank Adaptation) so we only train ~0.5% of the model's parameters.
This makes training fast and memory-efficient.

**Why SFT first?** RL can amplify existing capabilities but can't teach them from scratch.
Without SFT, the base model makes legal moves ~5% of the time. After SFT, it should reach >90%.

**Cost:** ~$3.75 (1x A100-80GB for ~1.5h at $2.50/hr)

In [ ]:
# Submit SFT training on Modal (1x A100 80GB, ~1-2 hours)
# Uses --detach so the job continues even if you close this notebook
!modal run --detach scripts/modal_app.py::run_sft

**Monitor progress:**
- Check your [Modal dashboard](https://modal.com/apps) for real-time logs
- Check your [W&B dashboard](https://wandb.ai) for the `chess-sft` run — look for `train/loss` decreasing steadily
- Run the cell below to check if the job is still running

In [ ]:
# Check running apps
!modal app list

In [ ]:
# Verify SFT checkpoint was saved
!modal volume ls rl-chess .rv/checkpoints/chess-sft/merged/ 2>/dev/null || echo "SFT not done yet — check Modal dashboard"

## 6. GRPO Training

Now the fun part — **reinforcement learning**. GRPO works like this:

1. For each position, generate **8 candidate moves** (the "group")
2. Score each move with our reward function (format + legality + Stockfish quality)
3. Compute **relative advantages** within the group (better moves get positive advantage)
4. Update the model to make high-advantage moves more likely

Unlike PPO, GRPO doesn't need a separate critic network — it uses the group statistics
as a baseline. This makes it simpler and more memory-efficient.

**Architecture:** We use vLLM in **colocate mode** — it shares GPU memory with the trainer
on a single A100 80GB.

**Cost:** ~$1.25 (1x A100-80GB for ~30 min)

In [ ]:
# Submit GRPO training on Modal (1x A100 80GB, ~20-30 min)
!modal run --detach scripts/modal_app.py::run_grpo

**Monitor progress:**
- Check your [Modal dashboard](https://modal.com/apps) for real-time logs
- W&B metrics to watch (`chess-grpo` run):
  - `rewards/format_reward/mean` — should hit 1.0 quickly
  - `rewards/legality_reward/mean` — should climb steadily
  - `rewards/quality_reward/mean` — slower improvement
  - `reward` — combined signal, should trend upward

In [ ]:
# Check running apps
!modal app list

In [ ]:
# Verify GRPO checkpoint was saved
!modal volume ls rl-chess .rv/checkpoints/chess-grpo/merged/ 2>/dev/null || echo "GRPO not done yet — check Modal dashboard"

## 7. Evaluation — Can We Beat GPT-4o?

Time to test our trained model:
1. **Puzzle accuracy** — legal move rate and Stockfish evaluation on held-out positions
2. **Full games vs GPT-4o** — play actual chess games and track wins/losses

**Cost:** ~$0.60 (1x A100-80GB for ~15 min)

In [ ]:
# Run evaluation on Modal (1x A100 80GB, ~15 min)
!modal run --detach scripts/modal_app.py::run_eval

In [ ]:
# Pull evaluation results from Modal volume
!modal run scripts/modal_app.py::fetch_results

In [ ]:
import json, subprocess

result = subprocess.run(
    ["modal", "run", "scripts/modal_app.py::fetch_results"],
    capture_output=True, text=True
)

if result.stdout.strip() and '{' in result.stdout:
    # Find the JSON in the output (skip any modal startup messages)
    lines = result.stdout.strip().split('\n')
    json_str = '\n'.join(l for l in lines if l.strip().startswith('{') or l.strip().startswith('"') or l.strip().startswith('}') or l.strip().startswith('['))
    data = json.loads(json_str)
    if "error" in data:
        print(data["error"])
    else:
        pr = data["puzzle_results"]
        gr = data["game_results"]
        print(f"Model: {data['model_path']}")
        print(f"\nPuzzle Evaluation ({pr['num_evaluated']} positions):")
        print(f"  Legal move rate: {pr['legal_rate']:.1%}")
        print(f"  Avg Stockfish score: {pr['avg_score']:.1f} cp")
        print(f"\nGames vs GPT-4o ({gr['total']} games):")
        print(f"  Wins: {gr['wins']}  Losses: {gr['losses']}  Draws: {gr['draws']}")
        print(f"  Win rate: {gr['win_rate']:.1%}")
        if gr.get("wins_by_opponent_illegal"):
            print(f"  ({gr['wins_by_opponent_illegal']} wins from opponent illegal moves)")
else:
    print("No eval results found yet — check Modal dashboard")

## 8. Play Against Your Model

Deploy the model as a vLLM server on Modal and play against it in the browser!

**Architecture:**
```
Browser (localhost:3000) → Next.js API route → vLLM on Modal (HTTPS)
```

**Cost:** ~$2.50/hr while the server is running (scales to zero when idle after 15 min)

In [ ]:
# Deploy vLLM server on Modal — this gives you a persistent HTTPS URL
!modal deploy scripts/modal_app.py

Copy the serve URL from the output above (looks like `https://your-workspace--rl-chess--serve.modal.run`).
Then update the web app config and start it:

```bash
# In a terminal:
cd web
echo 'VLLM_BASE_URL=https://YOUR-URL-HERE/v1' > .env.local
pnpm install && pnpm dev
# Open http://localhost:3000
```

> **Note:** The first request may take ~60s as Modal cold-starts the GPU container and loads the model.

In [ ]:
# When you're done playing, stop the deployment to avoid charges
!modal app stop rl-chess

## What's Next?

**Results** (SFT 30K examples + GRPO 1000 steps on H100):
- **100% legal move rate** on 100 held-out puzzles — our model never makes an illegal move
- **0W/5L/5D vs GPT-4o** with fair evaluation (10 retries per move, 200-move limit)
- GPT-4o needed **58 total retries** for illegal moves across 10 games — our model needed 0

**Total cost for this run:** ~$8-10 for the full pipeline on Modal (H100), plus ~$2.50/hr for serving.

### Future Research Directions

1. **Game-outcome rewards** — Instead of scoring individual moves with Stockfish, play full games during GRPO and reward wins/draws/losses. This gives the model a signal for multi-move planning and endgame conversion.

2. **Full game transcripts in SFT** — Include complete grandmaster games (from the Lichess game database, not just puzzles) so the model sees opening → middlegame → endgame → checkmate as a coherent arc.

3. **Game history in the prompt** — Pass the move history (or last N moves) alongside the FEN so the model can maintain strategic continuity across moves.

4. **Endgame tablebases** — For positions with ≤7 pieces, there's a mathematically perfect answer. Include these as training data for basic endgame technique (K+R vs K, pawn promotion, etc.).

5. **Smarter reward shaping** — Add a checkmate bonus, material advantage component, or weight endgame positions higher in the quality reward.

6. **Curriculum learning** — Start GRPO with easy endgame positions, then gradually increase difficulty.

7. **Scale up** — Larger base models (14B, 32B), more SFT data (30K → 100K+), more GRPO steps (1000 → 5000+), multi-GPU training.

8. **Better evaluation** — Play against Stockfish at various Elo levels, measure actual Elo rating, evaluate opening/middlegame/endgame separately.